In [ ]:
# ===========================================
# Notebook Name:
# 05_drop_legacy_tournament_columns
#
# Purpose:
# Drop the 14 legacy pokemon.silver.tournaments
# columns that were superseded by v2 columns
# during the v1->v2 migration (Step 1-4). This
# is a destructive, irreversible schema change
# on a live table and is NOT part of the
# Daily/Weekly Workflow.
#
# Legacy columns removed:
# event_title, event_type_id, event_type_title,
# event_date_display, prefecture_name, venue,
# address, shop_name, league, regulation,
# capacity, result_count, source_url,
# response_hash
#
# Preconditions (must both pass before running
# this notebook):
# 1. 00_migration/03_migrate_existing_data has
#    run at least once (backfills venue_name,
#    prefecture, event_type, event_date from
#    their legacy counterparts).
# 2. 00_migration/04_validate_v2_migration
#    "Check 5" passes with zero unbackfilled
#    rows for the 4 columns that have a v2
#    counterpart (venue, prefecture_name,
#    event_type_title, event_date_display).
#
# The other 10 legacy columns have no v2
# counterpart at all; they are dropped outright
# because a repo-wide search found no
# Gold/ML/Analysis/src consumer of them. The
# only notebook that reads several of these
# names (02_silver/01_identify_result_fetch_targets)
# already does so via existence checks
# (`if column_name in tournament_columns`) that
# degrade gracefully once the columns are gone.
#
# Run manually, once, after Check 5 passes.
# ===========================================

In [ ]:
TOURNAMENTS_TABLE = "pokemon.silver.tournaments"

LEGACY_COLUMNS_WITH_V2_COUNTERPART = {
    "venue": "venue_name",
    "prefecture_name": "prefecture",
    "event_type_title": "event_type",
    "event_date_display": "event_date",
}

LEGACY_COLUMNS_WITHOUT_V2_COUNTERPART = [
    "event_title",
    "event_type_id",
    "address",
    "shop_name",
    "league",
    "regulation",
    "capacity",
    "result_count",
    "source_url",
    "response_hash",
]

LEGACY_COLUMNS_TO_DROP = (
    list(LEGACY_COLUMNS_WITH_V2_COUNTERPART)
    + LEGACY_COLUMNS_WITHOUT_V2_COUNTERPART
)

print("Target table :", TOURNAMENTS_TABLE)
print("Columns to drop:", LEGACY_COLUMNS_TO_DROP)

In [ ]:
from pyspark.sql import functions as F

# -------------------------------------------
# Re-verify backfill completeness right before
# dropping, in case data has changed since
# 04_validate_v2_migration last ran. This is
# the same check as Check 5 in that notebook,
# repeated here so this notebook fails closed
# even if run out of order.
# -------------------------------------------
tournaments_df = spark.table(TOURNAMENTS_TABLE)

backfill_failures = []

for legacy_column, v2_column in LEGACY_COLUMNS_WITH_V2_COUNTERPART.items():
    unbackfilled_count = (
        tournaments_df
        .filter(
            F.col(legacy_column).isNotNull()
            & F.col(v2_column).isNull()
        )
        .count()
    )

    if unbackfilled_count > 0:
        print(
            f"{legacy_column} -> {v2_column}: "
            f"{unbackfilled_count} row(s) not backfilled"
        )
        backfill_failures.append(
            f"{legacy_column} -> {v2_column}: "
            f"{unbackfilled_count} row(s) not backfilled"
        )
    else:
        print(f"{legacy_column} -> {v2_column}: backfill complete.")

if backfill_failures:
    raise ValueError(
        "Refusing to drop legacy columns: "
        f"{len(backfill_failures)} backfill check(s) failed. "
        "Re-run 00_migration/03_migrate_existing_data first. "
        f"Details: {backfill_failures}"
    )

print("Backfill re-check passed. Safe to proceed.")

In [ ]:
# -------------------------------------------
# Delta column mapping must be enabled before
# DROP COLUMN is allowed.
# -------------------------------------------
spark.sql(f"""
ALTER TABLE {TOURNAMENTS_TABLE}
SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion' = '2',
    'delta.minWriterVersion' = '5'
)
""")

print("delta.columnMapping.mode enabled on", TOURNAMENTS_TABLE)

In [ ]:
# -------------------------------------------
# Drop the 14 legacy columns. Each column is
# dropped in its own ALTER TABLE statement so a
# partial failure leaves a clear error pointing
# at the specific column, rather than an
# all-or-nothing multi-column statement.
# -------------------------------------------
existing_columns = {
    field.name
    for field in spark.table(TOURNAMENTS_TABLE).schema.fields
}

for column_name in LEGACY_COLUMNS_TO_DROP:
    if column_name not in existing_columns:
        print(f"Skipping {column_name}: already absent.")
        continue

    spark.sql(
        f"ALTER TABLE {TOURNAMENTS_TABLE} "
        f"DROP COLUMN {column_name}"
    )
    print(f"Dropped column: {column_name}")

print("Legacy column drop complete.")

In [ ]:
# -------------------------------------------
# Verify: none of the 14 legacy columns remain,
# and no v2 column was accidentally dropped.
# -------------------------------------------
remaining_columns = {
    field.name
    for field in spark.table(TOURNAMENTS_TABLE).schema.fields
}

still_present = set(LEGACY_COLUMNS_TO_DROP) & remaining_columns

if still_present:
    raise ValueError(
        f"{len(still_present)} legacy column(s) still present "
        f"after drop: {sorted(still_present)}"
    )

REQUIRED_V2_COLUMNS = {
    "tournament_id",
    "venue_name",
    "prefecture",
    "event_type",
    "event_date",
    "event_hash",
}

missing_v2_columns = REQUIRED_V2_COLUMNS - remaining_columns

if missing_v2_columns:
    raise ValueError(
        f"v2 column(s) unexpectedly missing after drop: "
        f"{sorted(missing_v2_columns)}"
    )

print("Verification passed: legacy columns removed, v2 columns intact.")
display(
    spark.sql(f"DESCRIBE TABLE {TOURNAMENTS_TABLE}")
)